[NB-RECORDINGS] Recordings Colab: dataset subject + your own files | owner: tools-build-engineer | status: draft
# Your recordings and the dataset
**Gpo 604, Bioinstrumentation, EMG → ML**

By the end of this notebook you have one table and one figure that compare a dataset subject with your own recordings: size (RMS), spread across repetitions, and spectrum.

**How to work**
1. **File → Save a copy in Drive** (so your work is yours).
2. Go top to bottom. Press **▶** on a cell, or **Ctrl+Enter**.
3. Cells that start with **TYPE** are the ones where you change values. Cells that start with **RUN** you only run.
4. The same code also runs in plain Python (a `.py` file or Jupyter). Then your files go in a folder called `my_recordings/`.

**Units.** The dataset files hold *file units* (the file does not state a unit). Your own files hold *ADC counts* (whole numbers from the Arduino, with an offset).

---
## Step 0, Your group and your dataset subject

| | |
|:--|:--|
| **What** | Type your group number (1 to 7). A fixed draw table gives you one subject of the public dataset. |
| **Why** | Each group reads different dataset files, so you can compare notes. The draw is the same for everyone. |
| **You type** | `GROUP = 1` → your number. |
| **You will see** | `Group 1 -> dataset subject 20`. |

In [ ]:
# TYPE — Step 0: your group number
GROUP = 1                                        # <-- type your group number, 1 to 7

DRAW = {1: 20, 2: 12, 3: 3, 4: 15, 5: 4, 6: 18, 7: 1}   # group -> dataset subject (fixed draw)
assert GROUP in DRAW, "GROUP must be a number from 1 to 7."   # stop early if the number is wrong
SUBJECT = DRAW[GROUP]                            # the subject you will read
print(f"Group {GROUP} -> dataset subject {SUBJECT}")

---
## Step 1, Get the dataset

| | |
|:--|:--|
| **What** | Copy the public dataset from GitHub onto this computer. Nothing is saved on your laptop. |
| **Why** | Colab starts empty every session. No data, nothing to analyse. |
| **You will see** | Some text, then **`Dataset ready.`** and the working folder. |

In [ ]:
# RUN — Step 1: copy the dataset from GitHub (about 1 minute the first time)
import os, subprocess                                  # files and programs
REPO = "MuscleTracker-Hand-Movement"                   # folder name of the dataset
URL = "https://github.com/RQ-FA/MuscleTracker-Hand-Movement"   # where it lives
if not os.path.exists(REPO):                           # only download once
    subprocess.run(["git", "clone", "--depth", "1", URL], check=True)   # same as: git clone
print("Dataset ready.")
print("Working folder:", os.getcwd())                  # where your files go

---
## Step 2, Load the dataset subject

| | |
|:--|:--|
| **What** | Load six files of your subject: movement 2 (wrist flexion, **active**) and movement 5 (**rest**), repetitions 1 to 3, channel 1. |
| **Why** | The file name is the metadata: `subject_repetition_movement.txt`. Fs = 1000 Hz comes from the instrument, not from the data. |
| **You will see** | How many repetitions were found for each task. A few subjects have only 2: the notebook then uses 2 and says so. |

In [ ]:
# RUN — Step 2: load the dataset files (channel 1 = column 0)
import numpy as np                                     # arrays of numbers
import pandas as pd                                    # tables
import matplotlib.pyplot as plt                        # plots
import scipy as sp                                     # the FFT

DATA = os.path.join(REPO, "Data", "Raw", "Biopac")     # folder with the .txt files
FS_DATASET = 1000                                      # Hz, from the dataset paper (not computed)
records = []                                           # one entry per file, dataset and own

for task, movement in [("active", 2), ("rest", 5)]:    # movement 2 = flexion, 5 = rest
    found = 0                                          # how many repetitions exist
    for rep in (1, 2, 3):                              # repetitions 1 to 3
        name = f"{SUBJECT}_{rep}_{movement}.txt"       # subject_repetition_movement.txt
        path = os.path.join(DATA, name)                # full path of the file
        if not os.path.exists(path):                   # some subjects miss a repetition
            continue                                   # skip it
        x = np.loadtxt(path, delimiter=",")            # read the numbers
        s = x[:, 0] if x.ndim == 2 else x              # channel 1 = column 0
        records.append(dict(source="dataset", task=task, rep=rep, file=name, fs=FS_DATASET, s=s))
        found += 1                                     # count it
    print(f"{task}: {found} repetition(s) found" + ("" if found == 3 else "  (fewer than 3, using what exists)"))

---
## Step 3, Your own recordings

| | |
|:--|:--|
| **What** | Type the settings of your Arduino. Then give the notebook your files: `active_1.txt` ... `active_4.txt`, `rest_1.txt` ... `rest_4.txt` and `metadata.csv` (one row per file). Each file has one number per line, in ADC counts. |
| **Why** | The notebook needs Fs, the ADC size and the band to turn numbers into seconds, volts and Hz. The band comes from a source before you record (McManus 2020, course 20 to 450 Hz). |
| **You type** | `FS_MINE`, `ADC_BITS`, `VREF`, and the band if your group chose another one. |
| **You will see** | In Colab, a button to choose files. In plain Python, the folder `my_recordings/` is read. Then the list of files found. |

In [ ]:
# TYPE — Step 3: your own values (type your values)
FS_MINE = 1000        # Hz, your sampling frequency (type your value)
ADC_BITS = 10         # bits of your ADC, Arduino Uno = 10 (type your value)
VREF = 5.0            # V, the reference voltage of your ADC (type your value)
BW_LOW = 20           # Hz, low edge of your band (type your value)
BW_HIGH = 450         # Hz, high edge of your band (type your value)

FOLDER = "my_recordings"                               # your files live here
os.makedirs(FOLDER, exist_ok=True)                     # make the folder if it is missing
try:                                                   # Colab: ask for files
    from google.colab import files                     # only exists in Colab
    uploaded = files.upload()                          # opens the file chooser
    for fname, content in uploaded.items():            # every chosen file
        with open(os.path.join(FOLDER, fname), "wb") as fh:   # save it in the folder
            fh.write(content)                          # write the bytes
except ImportError:                                    # plain Python: no chooser
    print(f"Plain Python: reading the folder {FOLDER}/")

meta_path = os.path.join(FOLDER, "metadata.csv")       # the description of your files
if os.path.exists(meta_path):                          # read it if it exists
    meta = pd.read_csv(meta_path).set_index("file_name")   # one row per file name
else:
    meta = pd.DataFrame()                              # no metadata yet
    print("No metadata.csv found: the speed check cannot run.")

names = sorted(n for n in os.listdir(FOLDER)           # your recording files
               if n.startswith(("active_", "rest_")) and n.endswith(".txt"))
for name in names:                                     # load each one
    task = name.split("_")[0]                          # active or rest
    rep = int(name.split("_")[1].split(".")[0])        # repetition number
    s = np.loadtxt(os.path.join(FOLDER, name))         # one number per line
    records.append(dict(source="own", task=task, rep=rep, file=name, fs=FS_MINE, s=s))
print("Own files found:", names if names else "none yet")

---
## Step 4, Describe every file

| | |
|:--|:--|
| **What** | For every file: number of samples N, duration T = N / Fs, mean (the offset), RMS with the mean removed, share of power inside your band, minimum and maximum. |
| **Why** | The same few numbers tell you if a recording is usable. The RMS removes the mean first because your Arduino adds an offset (a constant) that is not muscle. |
| **Three checks (own files)** | **Zero level**: the mean of each rest file is your offset. **No clipping**: no sample at 0 or at 2**ADC_BITS - 1. **Right speed**: N equals planned duration x Fs within 2 percent. |
| **You will see** | The offsets, then one table with one row per file. `ok` means a check passed, `FAIL` means look at that file. |

In [ ]:
# RUN — Step 4: one row of numbers per file
def band_share(s, fs):                                 # share of power inside the band
    n = len(s)                                         # number of samples
    spec = np.abs(sp.fft.fft(s - s.mean())) / n        # magnitude, mean removed (course recipe)
    half = n // 2                                      # first half only (up to Fs/2)
    power = spec[:half] ** 2                           # power per frequency bin
    f = np.arange(half) * fs / n                       # bin number -> Hz
    inside = (f >= BW_LOW) & (f <= BW_HIGH)            # True inside the band
    return power[inside].sum() / power.sum()           # fraction, 0 to 1

rows = []                                              # table rows
for r in records:                                      # every file
    s, fs = r["s"], r["fs"]                            # signal and its Fs
    n = len(s)                                         # N
    row = dict(source=r["source"], task=r["task"], rep=r["rep"], file=r["file"])
    row.update(N=n, T_s=round(n / fs, 3))              # duration T = N / Fs
    row.update(mean=round(s.mean(), 3))                # the mean = the offset
    row.update(rms=np.sqrt(np.mean((s - s.mean())**2)))   # RMS with the mean removed
    row.update(band_share=round(band_share(s, fs), 3)) # power share in [BW_LOW, BW_HIGH]
    row.update(min=s.min(), max=s.max())               # smallest and largest sample
    row.update(clip_ok="-", speed_ok="-")              # checks only for own files
    if r["source"] == "own":                           # the three checks
        clipped = s.min() <= 0 or s.max() >= 2**ADC_BITS - 1   # a sample at the limit
        row["clip_ok"] = "FAIL" if clipped else "ok"   # no clipping
        if r["file"] in meta.index:                    # planned duration from metadata.csv
            plan = float(meta.loc[r["file"], "planned_duration_s"])   # seconds
            off = abs(n - plan * fs) / (plan * fs)     # relative difference
            row["speed_ok"] = "ok" if off <= 0.02 else "FAIL"   # within 2 percent
        else:
            row["speed_ok"] = "no metadata"            # cannot check
    rows.append(row)

table = pd.DataFrame(rows)                             # one row per file
for r in records:                                      # zero level: offset of each rest file
    if r["source"] == "own" and r["task"] == "rest":
        m = r["s"].mean()                              # offset in ADC counts
        print(f"Offset of {r['file']}: {m:.1f} counts = {m * VREF / (2**ADC_BITS - 1):.2f} V")
pd.set_option("display.width", 200)                    # wide table
print(table.round(3).to_string(index=False))           # the table, 3 decimals

---
## Step 5, Mean and spread across repetitions

| | |
|:--|:--|
| **What** | For each source (dataset, own) and task (active, rest): the mean and the standard deviation (SD) of the RMS over the repetitions. Then the ratio active / rest. |
| **Why** | One number from one repetition can mislead. The mean is the typical size. The SD shows how much repetitions differ. |
| **You will see** | A small table and one ratio per source. A ratio far above 1 means the active signal is clearly bigger than rest. |

In [ ]:
# RUN — Step 5: mean and SD of the RMS over repetitions
summary = (table.groupby(["source", "task"])["rms"]    # group the RMS values
           .agg(["count", "mean", "std"]).round(3))    # SD uses n - 1
print(summary)                                         # count, mean, SD

ratios = {}                                            # active / rest per source
for src in table["source"].unique():                   # dataset, own
    m = table[table.source == src].groupby("task")["rms"].mean()   # mean RMS per task
    if "active" in m and "rest" in m:                  # both tasks present
        ratios[src] = round(m["active"] / m["rest"], 2)   # ratio of the means
        print(f"{src}: mean RMS active / rest = {ratios[src]}")

---
## Step 6, One combined figure

| | |
|:--|:--|
| **What** | Left column: dataset subject. Right column: your files. Panel 1: traces of all repetitions on top of each other (active, rest) with the same y scale. Panel 2: RMS per repetition with the mean ± SD. Panel 3: mean spectrum of the active repetitions with a ± SD band. |
| **Why** | Traces show what the signal looks like. Bars show size and spread. The spectrum shows where the energy lives. |
| **Care** | To average spectra, every repetition is cut to the shortest N. The frequency step is Fs / N, so the same N gives the same frequency axis. |
| **You will see** | A figure saved as `combined_figure.png`. The right column says "no own files yet" if you gave no files. |

In [ ]:
# RUN — Step 6: the combined figure
COL = {"active": "#1F5FBF", "rest": "#2E8B57"}         # blue = active, green = rest
INK = "#1E2A26"                                        # text colour
spec_info = {}                                         # N used and frequency step per source

def tag(ax, text):                                     # panel number inside the frame
    ax.text(0.015, 0.93, text, transform=ax.transAxes, fontweight="bold", va="top", color=INK)

def empty(ax, src):                                    # panel without data
    ax.text(0.5, 0.5, "no own files yet", transform=ax.transAxes, ha="center", color="#5B6964")
    ax.set_xticks([]); ax.set_yticks([])               # no axes marks

def traces(ax, recs, task, unit, ylim):                # panel 1: repetitions overlaid
    for r in recs:                                     # every repetition of this task
        ax.plot(np.arange(len(r["s"])) / r["fs"], r["s"], lw=0.5, alpha=0.7, color=COL[task])
    ax.set_ylim(ylim); ax.set_ylabel(f"{task} ({unit})"); ax.set_xlabel("Time (s)")

def bars(ax, recs_by_task, src):                       # panel 2: RMS per repetition
    x0 = 0                                             # next bar position
    for task in ("active", "rest"):                    # two groups of bars
        vals = table[(table.source == src) & (table.task == task)]["rms"].to_numpy()
        xs = np.arange(x0, x0 + len(vals))             # one bar per repetition
        ax.bar(xs, vals, color=COL[task], label=task)  # RMS bars
        if len(vals):                                  # mean line and SD band
            m, sd = vals.mean(), vals.std(ddof=1) if len(vals) > 1 else 0.0
            ax.hlines(m, xs[0] - 0.4, xs[-1] + 0.4, color=INK, lw=2)         # mean
            ax.hlines([m - sd, m + sd], xs[0] - 0.4, xs[-1] + 0.4, color=INK, lw=1, ls="--")   # ± SD
        x0 += len(vals) + 1                            # gap between the groups
    ax.set_xticks([]); ax.set_xlabel("Repetitions (line = mean, dashed = ± SD)")
    ax.legend(loc="upper right", fontsize=8)           # inside the frame

def spectrum(ax, recs, src):                           # panel 3: mean spectrum ± SD
    nmin = min(len(r["s"]) for r in recs)              # shortest N
    fs = recs[0]["fs"]                                 # Fs of this source
    mags = []                                          # one spectrum per repetition
    for r in recs:                                     # cut to nmin, remove the mean
        s = r["s"][:nmin]                              # cut to the shortest N
        mags.append(np.abs(sp.fft.fft(s - s.mean())) / nmin)   # course recipe
    mags = np.array(mags)[:, : nmin // 2] ** 2         # first half, squared
    f = np.arange(nmin // 2) * fs / nmin               # Hz, step = Fs / N
    m = mags.mean(axis=0)                              # mean spectrum
    sd = mags.std(axis=0, ddof=1) if len(recs) > 1 else np.zeros_like(m)   # SD band
    ax.plot(f, m, color=COL["active"], lw=1)           # mean line
    ax.fill_between(f, np.maximum(m - sd, 0), m + sd, color=COL["active"], alpha=0.25)   # ± SD
    ax.axvline(BW_LOW, color=INK, ls=":", lw=1); ax.axvline(BW_HIGH, color=INK, ls=":", lw=1)
    ax.set_xlabel("Frequency (Hz)"); ax.set_ylabel("Power (units$^2$)")
    ax.set_title(f"active, cut to N = {nmin}, step = Fs/N = {fs / nmin:.3f} Hz", fontsize=8, pad=2)
    spec_info[src] = dict(N_used=int(nmin), freq_step_hz=round(fs / nmin, 4), n_rep=len(recs))

fig = plt.figure(figsize=(13, 12))                     # big figure
gs = fig.add_gridspec(4, 2, height_ratios=[1, 1, 1, 1.2])   # 4 rows x 2 columns
for c, (src, title, unit) in enumerate([("dataset", f"Dataset subject {SUBJECT}", "file units"),
                                         ("own", f"Your recordings (group {GROUP})", "ADC counts")]):
    recs = [r for r in records if r["source"] == src]  # this column's files
    axs = [fig.add_subplot(gs[i, c]) for i in range(4)]   # four panels
    axs[0].set_title(title, fontweight="bold", color=INK)
    if not recs:                                       # nothing to draw
        for a in axs: empty(a, src)
        continue
    allv = np.concatenate([r["s"] for r in recs])      # one y scale for the whole source
    pad = 0.05 * (allv.max() - allv.min())             # a little room
    ylim = (allv.min() - pad, allv.max() + pad)        # shared y limits
    for ax, task in zip(axs[:2], ("active", "rest")):  # panels 1a, 1b
        traces(ax, [r for r in recs if r["task"] == task], task, unit, ylim)
    bars(axs[2], recs, src); axs[2].set_ylabel(f"RMS ({unit})")   # panel 2
    act = [r for r in recs if r["task"] == "active"]   # active repetitions only
    if act: spectrum(axs[3], act, src)                 # panel 3
    for a, t in zip(axs, ("1a", "1b", "2", "3")): tag(a, t)   # numbers inside frame
fig.tight_layout()                                     # no overlaps
fig.savefig("combined_figure.png", dpi=130)            # save the picture
plt.show()                                             # show it
print("Saved: combined_figure.png in", os.getcwd())

---
## Step 7, What to do next

| | |
|:--|:--|
| **What** | Read your table and figure. Fix the files that show `FAIL`. |
| **Why** | A recording that fails a check cannot be compared. Record it again with the same duration. |
| **You will see** | Nothing new: this step is yours. Ask: is the active RMS clearly bigger than the rest RMS? Do the repetitions agree (small SD)? Is most power inside your band? |